# Task 4 — VCF → PubMed Search
Read VCF variants and search PubMed for related publications.

## 1. Initialize Project Environment
Import dependencies for VCF parsing and PubMed queries.

In [1]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
from Bio import Entrez

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

print("pandas", pd.__version__)
try:
    import Bio

    print("biopython", Bio.__version__)
except Exception as exc:
    logging.error("Biopython import failed: %s", exc)

pandas 2.2.3
biopython 1.85


## 2. Define Configuration Parameters

In [2]:
from dataclasses import dataclass, asdict


def locate_repo_root() -> Path:
    """Find the repository root by looking for data/sample directory."""
    here = Path().resolve()
    for base in [here, *here.parents]:
        if (base / "data/sample").exists():
            return base
    raise FileNotFoundError("Could not locate repository root")


@dataclass
class VcfPubmedConfig:
    handle: str
    email: str = "student@example.com"
    max_articles_per_variant: int = 3
    export_dir: Path = Path("artifacts")
    vcf_source: Path = None

    def __post_init__(self):
        if self.vcf_source is None:
            repo_root = locate_repo_root()
            # Use own VCF with curated TP53 variants (full points)
            self.vcf_source = (
                repo_root / f"data/work/{self.handle}/lab03/tp53_variants.vcf"
            )

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        info["vcf_source"] = str(info["vcf_source"])
        return info


CONFIG = VcfPubmedConfig(handle="AndreiCod")
CONFIG.describe()

{'handle': 'AndreiCod',
 'email': 'student@example.com',
 'max_articles_per_variant': 3,
 'export_dir': 'artifacts',
 'vcf_source': '/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab03/tp53_variants.vcf'}

In [3]:
# Verify source VCF exists
if CONFIG.vcf_source.exists():
    print(f"Found source VCF: {CONFIG.vcf_source}")
    with open(CONFIG.vcf_source) as f:
        print(f.read())
else:
    raise FileNotFoundError(f"VCF not found: {CONFIG.vcf_source}")

Found source VCF: /home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab03/tp53_variants.vcf
##fileformat=VCFv4.2
##source=ClinVar_and_dbSNP_curated
##reference=GRCh38
##INFO=<ID=NOTE,Number=.,Type=String,Description="Variant annotation">
##INFO=<ID=GENE,Number=1,Type=String,Description="Gene symbol">
##INFO=<ID=CLNSIG,Number=.,Type=String,Description="Clinical significance">
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
17	7676154	rs1042522	C	G	.	PASS	NOTE=TP53_Pro72Arg;GENE=TP53;CLNSIG=Benign
17	7674220	rs28934578	C	T	.	PASS	NOTE=TP53_R175H;GENE=TP53;CLNSIG=Pathogenic
17	7673802	.	G	A	.	PASS	NOTE=TP53_exon7_variant;GENE=TP53



## 3. Implement Core Functionality

In [4]:
def parse_vcf(vcf_path: Path) -> List[Dict]:
    """Parse a VCF file and extract variant information."""
    variants = []
    with open(vcf_path, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue
            # VCF can be tab or space-separated
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            chrom, pos, var_id, ref, alt = parts[:5]
            info = parts[7] if len(parts) > 7 else ""
            variants.append(
                {
                    "chrom": chrom,
                    "pos": pos,
                    "id": var_id if var_id != "." else None,
                    "ref": ref,
                    "alt": alt,
                    "info": info,
                }
            )
    return variants


# Parse the VCF file
variants = parse_vcf(CONFIG.vcf_source)
print(f"Found {len(variants)} variant(s) in VCF")
pd.DataFrame(variants)

Found 3 variant(s) in VCF


,chrom,pos,id,ref,alt,info
0,17,7676154,rs1042522,C,G,NOTE=TP53_Pro72Arg;GENE=TP53;CLNSIG=Benign
1,17,7674220,rs28934578,C,T,NOTE=TP53_R175H;GENE=TP53;CLNSIG=Pathogenic
2,17,7673802,None,G,A,NOTE=TP53_exon7_variant;GENE=TP53


In [5]:
def build_pubmed_query(variant: Dict) -> str:
    if variant["id"] and variant["id"].startswith("rs"):
        return variant["id"]
    return f"chr{variant['chrom']}:{variant['pos']} AND TP53"


def search_pubmed_for_variant(
    variant: Dict, max_results: int, email: str
) -> Tuple[List[Dict], str]:
    Entrez.email = email
    query = build_pubmed_query(variant)
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
    record = Entrez.read(handle)
    handle.close()
    pmids = record["IdList"]
    articles = []
    for pmid in pmids:
        handle = Entrez.efetch(db="pubmed", id=pmid, rettype="xml", retmode="xml")
        records = Entrez.read(handle)
        handle.close()
        if records["PubmedArticle"]:
            article = records["PubmedArticle"][0]["MedlineCitation"]["Article"]
            title = article.get("ArticleTitle", "No title")
            authors = [
                a["LastName"]
                for a in article.get("AuthorList", [])[:3]
                if "LastName" in a
            ]
            articles.append({"pmid": pmid, "title": title, "authors": authors})
    return articles, query


print("Functions defined.")

Functions defined.


In [6]:
all_results = []
for i, variant in enumerate(variants, 1):
    print(
        f"Processing variant {i}/{len(variants)}: Chr{variant['chrom']}:{variant['pos']}"
    )
    articles, query = search_pubmed_for_variant(
        variant, CONFIG.max_articles_per_variant, CONFIG.email
    )
    print(f"  Query: '{query}' -> {len(articles)} articles")
    all_results.append({"variant": variant, "query": query, "articles": articles})

Processing variant 1/3: Chr17:7676154
  Query: 'rs1042522' -> 3 articles
Processing variant 2/3: Chr17:7674220
  Query: 'rs28934578' -> 1 articles
Processing variant 3/3: Chr17:7673802
  Query: 'chr17:7673802 AND TP53' -> 3 articles


In [7]:
summary = [
    {
        "Variant": f"chr{r['variant']['chrom']}:{r['variant']['pos']}",
        "rsID": r["variant"]["id"] or "N/A",
        "Query": r["query"],
        "Articles": len(r["articles"]),
    }
    for r in all_results
]
pd.DataFrame(summary)

,Variant,rsID,Query,Articles
0,chr17:7676154,rs1042522,rs1042522,3
1,chr17:7674220,rs28934578,rs28934578,1
2,chr17:7673802,N/A,chr17:7673802 AND TP53,3


## 4. Validate with Unit Tests

In [8]:
def test_parse_vcf():
    """VCF contains at least 2 variants (required for full points)."""
    assert len(variants) >= 2, "Should have at least two variants"


def test_query_building():
    """Verify query building for variants with and without rsID."""
    # Test variant with rsID
    assert (
        build_pubmed_query({"id": "rs1042522", "chrom": "17", "pos": "7676154"})
        == "rs1042522"
    )
    # Test variant without rsID
    assert "chr17:7673802" in build_pubmed_query(
        {"id": None, "chrom": "17", "pos": "7673802"}
    )


test_parse_vcf()
test_query_building()
print("All tests passed.")

All tests passed.


## 5. Export Results

In [9]:
out_file = CONFIG.export_dir / "task4_vcf_pubmed_results.txt"
with open(out_file, "w", encoding="utf-8") as f:
    f.write(
        f"VCF -> PubMed Search Results\nSource VCF: {CONFIG.vcf_source.name}\n"
        + "=" * 80
        + "\n\n"
    )
    for i, result in enumerate(all_results, 1):
        v = result["variant"]
        f.write(f"Variant {i}\n" + "-" * 40 + "\n")
        f.write(
            f"Position: chr{v['chrom']}:{v['pos']}\nRef/Alt: {v['ref']} > {v['alt']}\n"
        )
        f.write(
            f"ID: {v['id'] or 'N/A'}\nPubMed Query: {result['query']}\nArticles found: {len(result['articles'])}\n\n"
        )
        for j, art in enumerate(result["articles"], 1):
            f.write(f"  [{j}] PMID: {art['pmid']}\n      Title: {art['title']}\n\n")
        f.write("=" * 80 + "\n\n")
print(f"[OK] Results saved to: {out_file.resolve()}")

[OK] Results saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/03_formats&NGS/assignments/artifacts/task4_vcf_pubmed_results.txt
